---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md)
*Search [existing issues](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues) first.*

### 💾 Save your own copy
> **File → Save a copy in Drive** — saves a personal editable copy to your Google Drive.
> Do this before making edits, otherwise changes are lost when the session ends.

# 📉 Principal Components Regression (PCR)
**ISLP Chapter 6 · Pattern Recognition for the Rest of Us**

> When predictors are highly correlated, OLS becomes unstable. PCR solves this by compressing X into a small set of uncorrelated components, then regressing on those components instead.

### Dataset: Capital Bikeshare — Washington DC (2011–2012)
8,645 hourly records of bike rentals from a public bike-sharing system.
**Business / public policy application:** urban transportation demand forecasting,
infrastructure planning, climate impact on mobility.

`temp` and `atemp` have r = 0.99 — a textbook example of multicollinearity
that destabilises OLS coefficients. PCR handles this directly.

### What you'll learn
- Why multicollinearity breaks OLS and how PCR fixes it
- How PCA components are constructed and what they represent
- Choosing the number of components M via cross-validation
- Back-transforming PCR coefficients to original feature space

### Time: ~45 minutes

In [ ]:
# ⚠️  Run this cell first — sets up data and imports
# Tip: Runtime → Run all  (runs everything top to bottom)

import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
    'axes.grid': True, 'grid.alpha': 0.4,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11
})

# ── Load Capital Bikeshare dataset (Washington DC, 2011-2012) ─────────────────
# Source: ISLP Chapter 7 / UCI ML Repository
# 8,645 hourly records — predict total bike rentals (bikers) from weather & time
# Key multicollinearity: temp & atemp (r = 0.99) — apparent temperature mirrors
# actual temperature, making OLS coefficients unstable

bikeshare = pd.read_csv('https://raw.githubusercontent.com/ladataanalytics/pattern-recognition-notebooks/main/data/Bikeshare.csv')
print(f'✓ Bikeshare: {bikeshare.shape}')

# Encode categoricals as numeric
bikeshare['mnth_num']    = pd.Categorical(bikeshare['mnth']).codes
bikeshare['weather_num'] = pd.Categorical(bikeshare['weathersit']).codes

# Features and target
features = ['temp','atemp','hum','windspeed',
            'hr','season','weekday','workingday',
            'holiday','mnth_num','weather_num']
target   = 'bikers'
p        = len(features)

X = bikeshare[features].values.astype(float)
y = bikeshare[target].values.astype(float)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

scaler  = StandardScaler().fit(X_tr)
X_tr_s  = scaler.transform(X_tr)
X_te_s  = scaler.transform(X_te)

print(f'  Features ({p}): {features}')
print(f'  Target: hourly bike rentals, mean={y.mean():.1f}, range={y.min():.0f}–{y.max():.0f}')
print(f'  Train: {X_tr_s.shape}  Test: {X_te_s.shape}')
print(f'  Python {sys.version.split()[0]} | pandas {pd.__version__}')
print('✓ Setup complete')


## 📐 Part 1 — The Multicollinearity Problem

OLS finds β by solving **(XᵀX)β = Xᵀy**. When XᵀX is near-singular (correlated predictors), small data changes produce wildly different coefficients — high variance.

**The Bikeshare problem:** `temp` (actual temperature) and `atemp` (perceived temperature) move together almost perfectly (r = 0.99). OLS cannot reliably separate their individual effects.

**PCR fix:**
1. Standardise X
2. Run PCA → find M orthogonal components maximising variance in X
3. Project X → Z (n × M, uncorrelated by construction)
4. OLS on Z → ZᵀZ is well-conditioned regardless of X correlations

In [ ]:
# ── Multicollinearity in the Bikeshare dataset ────────────────────────────────
df_corr = pd.DataFrame(X_tr_s, columns=features)
corr    = df_corr.corr()

pairs = [(features[i], features[j], corr.iloc[i,j])
         for i in range(p) for j in range(i+1, p)]
pairs.sort(key=lambda x: -abs(x[2]))

print("Feature correlations (sorted by |r|):")
print(f"{'Feature A':12s}  {'Feature B':12s}  {'r':>7}")
print("-" * 36)
for a, b, r in pairs[:8]:
    bar = '█' * int(abs(r) * 25)
    print(f"{a:12s}  {b:12s}  {r:>7.3f}  {bar}")

cond = np.linalg.cond(X_tr_s.T @ X_tr_s)
print(f"\nCondition number of XᵀX: {cond:,.1f}")
print(f"  (Rule of thumb: > 1,000 = problematic)")

# Show OLS instability: refit on bootstrap samples, plot coefficient spread
np.random.seed(42)
n_boots  = 100
boot_coefs = []
for _ in range(n_boots):
    idx = np.random.choice(len(X_tr_s), len(X_tr_s), replace=True)
    c   = LinearRegression().fit(X_tr_s[idx], y_tr[idx]).coef_
    boot_coefs.append(c)
boot_coefs = np.array(boot_coefs)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Heatmap
im = axes[0].imshow(corr.values, cmap='RdYlGn', vmin=-1, vmax=1)
axes[0].set_xticks(range(p)); axes[0].set_xticklabels(features, rotation=45, ha='right', fontsize=8)
axes[0].set_yticks(range(p)); axes[0].set_yticklabels(features, fontsize=8)
for i in range(p):
    for j in range(p):
        axes[0].text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=6)
plt.colorbar(im, ax=axes[0], shrink=0.8)
axes[0].set_title('Feature Correlation Matrix\ntemp & atemp: r = 0.99')

# Bootstrap coefficient spread — shows instability for temp & atemp
axes[1].boxplot(boot_coefs, labels=features, patch_artist=True,
                boxprops=dict(facecolor='#aec6e8'),
                medianprops=dict(color='#1e5fa8', lw=2))
axes[1].axhline(0, color='grey', lw=0.8)
axes[1].set_xticklabels(features, rotation=45, ha='right', fontsize=8)
axes[1].set_ylabel('OLS coefficient (100 bootstrap samples)')
axes[1].set_title('OLS Coefficient Instability\nWide boxes = unstable due to multicollinearity')

plt.tight_layout()
plt.show()

print("\n📌 temp and atemp: huge box spread → OLS cannot distinguish their effects.")
print("   Their individual coefficients change sign across bootstrap samples.")
print("   PCR avoids this by working in component space where features are uncorrelated.")


## 🔬 Part 2 — PCA Step: Explained Variance

PCA finds M orthogonal directions (principal components) capturing maximum variance in X.
The first component likely captures the temp/atemp axis — the dominant source of variation.

In [ ]:
# ── PCA on standardised training data ────────────────────────────────────────
pca    = PCA().fit(X_tr_s)
cumvar = np.cumsum(pca.explained_variance_ratio_)
m90    = int(np.searchsorted(cumvar, 0.90)) + 1
m95    = int(np.searchsorted(cumvar, 0.95)) + 1

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(range(1, p+1), pca.explained_variance_ratio_ * 100,
            color='#1e5fa8', alpha=0.8, label='Per-component')
axes[0].plot(range(1, p+1), cumvar * 100, 'o-',
             color='#e85d20', lw=2, markersize=5, label='Cumulative')
axes[0].axhline(90, color='grey', lw=1, ls='--', label='90%')
axes[0].axvline(m90, color='#1a7a45', lw=1.5, ls='--', label=f'M={m90} → 90%')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Explained (%)')
axes[0].set_title('Scree Plot — Bikeshare Features')
axes[0].legend(fontsize=9)

axes[1].fill_between(range(1, p+1), cumvar * 100, alpha=0.15, color='#1e5fa8')
axes[1].plot(range(1, p+1), cumvar * 100, 'o-', color='#1e5fa8', lw=2)
axes[1].axhline(90, color='#e85d20', lw=1.5, ls='--', label='90%')
axes[1].axhline(95, color='#1a7a45', lw=1.5, ls='--', label='95%')
axes[1].set_xlabel('Number of Components M')
axes[1].set_ylabel('Cumulative Variance (%)')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend(fontsize=9)

plt.suptitle('PCA on Bikeshare Predictors', fontsize=12)
plt.tight_layout()
plt.show()

print(f"Components for 90% variance in X: M = {m90}")
print(f"Components for 95% variance in X: M = {m95}")
print()

# Show PC1 loadings — should be dominated by temp/atemp
loadings = pd.Series(np.abs(pca.components_[0]), index=features).sort_values(ascending=False)
print("PC1 loadings (|value|) — what drives the first component:")
for fname, v in loadings.items():
    bar = '█' * int(v * 35)
    print(f"  {fname:12s}: {v:.3f}  {bar}")
print()
print("📌 PC1 is dominated by temp and atemp — they move together almost identically.")
print("   PCR replaces these two correlated features with one well-defined component.")


## 📊 Part 3 — PCR: Choosing M by Cross-Validation

Fit OLS on the first M principal components. Choose M by cross-validation.

**Pipeline:** StandardScaler → PCA(n_components=M) → LinearRegression

In [ ]:
# ── PCR: cross-validate over M = 1 … p ──────────────────────────────────────
M_range = range(1, p + 1)
cv_mses = []
for M in M_range:
    pipe = Pipeline([('pca', PCA(n_components=M)), ('ols', LinearRegression())])
    mse  = -cross_val_score(pipe, X_tr_s, y_tr, cv=10,
                             scoring='neg_mean_squared_error').mean()
    cv_mses.append(mse)

best_M   = list(M_range)[np.argmin(cv_mses)]
best_mse = min(cv_mses)
ols_mse  = -cross_val_score(LinearRegression(), X_tr_s, y_tr, cv=10,
                              scoring='neg_mean_squared_error').mean()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(list(M_range), cv_mses, 'o-', color='#1e5fa8', lw=2,
        markersize=6, label='PCR 10-fold CV MSE')
ax.axhline(ols_mse, color='grey', lw=1.5, ls='--',
           label=f'Full OLS (M=p={p}) CV MSE = {ols_mse:.1f}')
ax.axvline(best_M, color='#e85d20', lw=2, ls='--',
           label=f'Best M = {best_M}  (CV MSE = {best_mse:.1f})')
ax.set_xlabel('Number of PCA Components M')
ax.set_ylabel('10-fold CV MSE')
ax.set_title('PCR: Choosing M via Cross-Validation — Bikeshare Dataset')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

pcr_best     = Pipeline([('pca', PCA(n_components=best_M)),
                          ('ols', LinearRegression())]).fit(X_tr_s, y_tr)
ols_full     = LinearRegression().fit(X_tr_s, y_tr)
pcr_test_mse = mean_squared_error(y_te, pcr_best.predict(X_te_s))
ols_test_mse = mean_squared_error(y_te, ols_full.predict(X_te_s))

pca_fit    = pcr_best.named_steps['pca']
var_kept   = pca_fit.explained_variance_ratio_.sum()

print(f"Best M = {best_M} components ({var_kept:.1%} of variance in X retained)")
print()
print(f"{'Model':25s}  {'CV MSE':>8}  {'Test MSE':>10}  {'Test RMSE':>10}")
print("-" * 58)
print(f"{'OLS (all features)':25s}  {ols_mse:>8.1f}  {ols_test_mse:>10.1f}  {ols_test_mse**0.5:>10.2f}")
print(f"{'PCR (M='+str(best_M)+')':25s}  {best_mse:>8.1f}  {pcr_test_mse:>10.1f}  {pcr_test_mse**0.5:>10.2f}")
print()
print("📌 RMSE is in bike-rental units — e.g. RMSE=80 means predictions are")
print("   off by ~80 rentals per hour on average.")


## ✅ Part 4 — Interpreting PCR Coefficients

PCR returns coefficients in **component space**. To interpret in terms of original features, back-transform:
`β_original = V @ β_pcr` where V is the p × M loading matrix from PCA.

In [ ]:
# ── Back-transform PCR coefficients to original feature space ────────────────
pca_fit  = pcr_best.named_steps['pca']
ols_fit  = pcr_best.named_steps['ols']

beta_orig = pca_fit.components_[:best_M].T @ ols_fit.coef_

# OLS for comparison
ols_sm = sm.OLS(y_tr, sm.add_constant(X_tr_s)).fit()

print(f"PCR implied coefficients vs OLS (M={best_M} components):")
print(f"{'Feature':12s}  {'PCR β':>10}  {'OLS β':>10}  {'p-value':>10}  note")
print("-" * 60)
for fname, b_pcr, b_ols, pval in zip(
        features, beta_orig, ols_sm.params[1:], ols_sm.pvalues[1:]):
    sig  = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''
    diff = '←differs' if abs(b_pcr - b_ols) > 10 else ''
    print(f"{fname:12s}  {b_pcr:>10.3f}  {b_ols:>10.3f}  {pval:>10.4f} {sig:3s} {diff}")

print()
print("📌 temp and atemp: PCR combines them into one component.")
print("   Their individual PCR coefficients are more stable than OLS.")
print("   OLS p-values show which features are statistically significant —")
print("   PCR doesn't provide p-values; use sm.OLS on the selected features")
print("   for inference after PCR guides you to the right dimensionality.")


## 💼 Exercise

**Context:** You're advising Washington DC's Department of Transportation on bike-sharing expansion. They want to predict peak-hour demand to plan dock capacity.

**Task 1 — Seasonal analysis:** Split the data by season (spring/summer/fall/winter). Fit PCR separately on each season. Does the optimal M differ? Which season is hardest to predict?

**Task 2 — Weather sensitivity:** Remove `atemp` from the feature set (keeping `temp`) and refit PCR. Does CV MSE change? What does this tell you about the atemp/temp redundancy?

**Task 3 — Compare to Ridge:** Fit `RidgeCV` on `X_tr_s`. Compare test RMSE to PCR. Which would you recommend to the transportation department — and why?

In [ ]:
# ── Exercise workspace ───────────────────────────────────────────────────────
# Variables: bikeshare, X_tr_s, X_te_s, y_tr, y_te, features, p
#            pcr_best, best_M, ols_full, pcr_test_mse, ols_test_mse

# Task 2 — Remove atemp, refit
no_atemp = [f for f in features if f != 'atemp']
idx      = [features.index(f) for f in no_atemp]
pipe_no  = Pipeline([('pca', PCA(n_components=best_M)),
                     ('ols', LinearRegression())])
mse_no   = mean_squared_error(y_te,
           pipe_no.fit(X_tr_s[:,idx], y_tr).predict(X_te_s[:,idx]))
print(f"PCR with all features (M={best_M}):    RMSE = {pcr_test_mse**0.5:.2f}")
print(f"PCR without atemp    (M={best_M}):    RMSE = {mse_no**0.5:.2f}")
print(f"Difference: {abs(pcr_test_mse**0.5 - mse_no**0.5):.2f} rentals/hr")
print()

# Task 3 — Ridge comparison
ridge = RidgeCV(alphas=np.logspace(-3, 4, 100), cv=10).fit(X_tr_s, y_tr)
ridge_mse = mean_squared_error(y_te, ridge.predict(X_te_s))
print(f"Ridge (λ={ridge.alpha_:.3f}): RMSE = {ridge_mse**0.5:.2f}")
print(f"PCR (M={best_M}):           RMSE = {pcr_test_mse**0.5:.2f}")
print(f"OLS:                        RMSE = {ols_test_mse**0.5:.2f}")


In [ ]:
# @title 📝 Quiz — Principal Components Regression
# @markdown Answer each question, then run this cell.

# @markdown **Q1:** Why must features be standardised before PCR?
# @markdown **Q2:** What property of principal components makes them immune to multicollinearity?
# @markdown **Q3:** When M = p in PCR, what model does it reduce to?
# @markdown **Q4:** Why doesn't PCR provide p-values for original features?
# @markdown **Q5:** The Bikeshare dataset has r=0.99 between temp and atemp. How does PCR handle this versus OLS?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the submission cell for AI feedback")


In [ ]:
_NB_NAME_ = "Principal Components Regression"
# @title 📋 Quiz Submission
# @markdown **Click ▶ Run** → copy the output → paste into Gemini in this Colab session.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(f"Q{i}: {str(v).strip() or '(no answer)'}"
                    for i, (_, v) in enumerate(answers.items(), 1))
    print(f'Please grade my quiz answers for the "{_NB_TITLE}" notebook')
    print(f'from "Data Pattern Recognition for the Rest of Us" (based on ISLP).')
    if GITHUB_USERNAME.strip(): print(f"Student: @{GITHUB_USERNAME.strip()}")
    print(); print(qa); print()
    print("For each: CORRECT/PARTIAL/INCORRECT, 2-3 sentence explanation,")
    print("ideal answer, Part to revisit if wrong.")
    print("End with: Overall grade + short study plan.")


---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md)

---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*